In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing  import OrdinalEncoder

In [3]:
df=pd.read_csv('dummy_covid.csv')

In [4]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [37]:
df['city'].value_counts()

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

In [5]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [6]:
from sklearn.model_selection  import train_test_split
X_train,X_test,y_train,y_trst=train_test_split(df.drop(columns=['has_covid']),df['has_covid'],test_size=0.2)

In [10]:
X_train

,age,gender,fever,cough,city
2,42,Male,101.0,Mild,Delhi
89,46,Male,103.0,Strong,Bangalore
71,75,Female,104.0,Strong,Delhi
84,69,Female,98.0,Strong,Mumbai
58,23,Male,98.0,Strong,Mumbai
...,...,...,...,...,...
27,33,Female,102.0,Strong,Delhi
37,55,Male,100.0,Mild,Kolkata
1,27,Male,100.0,Mild,Delhi
83,17,Female,104.0,Mild,Kolkata


# 1 Normal Method

In [12]:
# adding simple imputer to the fever col
si=SimpleImputer()
X_train_fever=si.fit_transform(X_train[['fever']])

# also the test data
X_test_fever=si.transform(X_test[['fever']])

In [17]:
X_train_fever.shape

(80, 1)

In [19]:
X_test_fever.shape

(20, 1)

In [23]:
# Ordinalencoding -> cough
oe=OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough=oe.fit_transform(X_train[['cough']])

# also the test data
X_test_cough=oe.transform(X_test[['cough']])

In [22]:
# OneHotEncoding -> gender, city
ohe=OneHotEncoder(drop='first',sparse_output=False)
X_train_gender_city=ohe.fit_transform(X_train[['gender','city']])

# also the test data
X_test_gender_city=ohe.transform(X_test[['gender','city']])

In [31]:
X_train_gender_city

array([[1., 1., 0., 0.],
       [1., 0., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 0., 1.],
       [1., 0., 0., 1.],
       [0., 0., 1., 0.],
       [0., 0., 0., 1.],
       [0., 0., 1., 0.],
       [1., 0., 1., 0.],
       [0., 0., 1., 0.],
       [0., 0., 1., 0.],
       [0., 0., 0., 1.],
       [1., 0., 1., 0.],
       [0., 0., 1., 0.],
       [1., 0., 0., 0.],
       [0., 0., 1., 0.],
       [0., 0., 0., 1.],
       [1., 0., 0., 0.],
       [1., 1., 0., 0.],
       [1., 0., 0., 0.],
       [1., 0., 1., 0.],
       [0., 0., 0., 0.],
       [0., 1., 0., 0.],
       [1., 0., 0., 1.],
       [1., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [1., 1., 0., 0.],
       [0., 0., 1., 0.],
       [1., 0., 0., 0.],
       [0., 0., 1., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 1.],
       [0., 0., 1., 0.],
       [0., 0., 1., 0.],
       [0., 0., 0., 1.],
       [1., 0., 0., 0.],
       [0., 1., 0., 0.],


In [30]:
X_test_gender_city.shape

(20, 4)

In [42]:
# Extracting Age
X_train_age=X_train.drop(columns=['gender','fever','cough','city']).values

# also the test data
X_test_age=X_test.drop(columns=['gender','fever','cough','city']).values

X_train_age.shape, X_test_age.shape

((80, 1), (20, 1))

In [46]:
X_train_transformed=np.concatenate((X_train_age, X_train_fever, X_train_gender_city, X_train_cough),axis=1)
# also the test data
X_test_transformed=np.concatenate((X_test_age, X_test_fever, X_test_gender_city, X_test_cough), axis=1)

X_train_transformed.shape, X_test_transformed.shape

((80, 7), (20, 7))

# 2 Column Transformer Method

In [47]:
from sklearn.compose import ColumnTransformer

In [48]:
transformer=ColumnTransformer(transformers=[
            ('tnf1',SimpleImputer(),['fever']),
            ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
            ('tnf3',OneHotEncoder(sparse_output=False, drop='first'),['gender','city'])
],remainder='passthrough')

In [49]:
transformer.fit_transform(X_train).shape

(80, 7)

In [50]:
transformer.transform(X_test).shape

(20, 7)